- 원래 교재 및 학교에선 배울땐 텐서플로우를 사용함
- 파이토치는 내가 임의로 추가한거

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

In [2]:
# 텐서로 변환하고 -1~1 범위로 정규화하는 변환기 정의
transform = transforms.Compose([
    transforms.ToTensor(), # 이미지를 텐서로 변환
    transforms.Normalize((0.5,), (0.5,)) # (평균, 표준편차)로 정규화
])

In [3]:
# 학습 데이터셋 및 데이터로더 불러오기
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# 테스트 데이터셋 및 데이터로더 불러오기
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 입력층: 784 (28*28)
        self.fc1 = nn.Linear(784, 128)
        # 은닉층
        self.fc2 = nn.Linear(128, 64)
        # 출력층: 10 (0~9)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        # 이미지를 1차원 벡터로 펼치기 (flatten)
        x = x.view(-1, 28 * 28)
        # 순전파 과정
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x) # 출력층에는 활성화 함수를 사용하지 않습니다.
        return x


In [6]:
# 모델, 손실 함수, 옵티마이저 정의

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 학습 루프
num_epochs = 3
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        # 순전파
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # 역전파
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'에포크 [{epoch+1}/{num_epochs}], 손실: {running_loss/len(train_loader):.4f}')
print("학습 완료!")


cuda
에포크 [1/3], 손실: 0.3997
에포크 [2/3], 손실: 0.1928
에포크 [3/3], 손실: 0.1440
학습 완료!


In [12]:
model.eval()

correct = 0
total = 0
with torch.no_grad(): # 평가 단계에서는 기울기 계산이 필요 없으므로 비활성화
    for data in test_loader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'\n테스트 데이터셋에서 모델의 정확도: {100 * correct / total:.2f}%')



테스트 데이터셋에서 모델의 정확도: 95.13%


In [15]:
inputs, labels = data
inputs

tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        ...,


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., 